# Week 5 — PokeAPI → pandas (beginner assignment)

This notebook builds a small dataset from [PokeAPI](https://pokeapi.co/), then inspects and analyzes it with pandas. Run cells **top to bottom** (needs internet).

## STEP 0 — Libraries

`requests` downloads JSON from the web. `pandas` turns rows into a `DataFrame`.

In [ ]:
import requests
import pandas as pd

## STEP 1a — Ask the API for a list of Pokémon

We use `pokemon?limit=20` so we get **20** Pokédex entries and a **URL** for each one’s details.

In [ ]:
# I'm asking: what endpoint gives me the first 20 Pokémon as a list?
# The result tells me how many items came back and gives me URLs for the next step — no skipped rows.
LIST_URL = "https://pokeapi.co/api/v2/pokemon?limit=20"
list_response = requests.get(LIST_URL)
list_response.raise_for_status()
list_data = list_response.json()
pokemon_list = list_data["results"]
len(pokemon_list)

## STEP 1b — Look at one list item

Each item has `name` and `url` pointing to the **full** Pokémon record.

In [ ]:
# I'm asking: what does a single search result look like so I know what to loop over?
# The result tells me I should use ["url"] for the detail request and ["name"] as a backup label.
pokemon_list[0]

## STEP 1c — Fetch **one** Pokémon’s details (trial)

This checks the JSON shape before we repeat it 20 times.

In [ ]:
# I'm asking: for one Pokémon, where are name, height, weight, base_experience, and the first type?
# The result tells me the exact keys so I can pull the same fields for every Pokémon.
trial_url = pokemon_list[0]["url"]
trial_response = requests.get(trial_url)
trial_response.raise_for_status()
p0 = trial_response.json()
p0["name"], p0["height"], p0["weight"], p0["base_experience"], p0["types"][0]["type"]["name"]

## STEP 1d — Fetch details for **each** of the 20 Pokémon

We append one dictionary per Pokémon: `name`, `height`, `weight`, `base_experience`, `first_type`.

**Units (from the API):** `height` is in **decimeters**; `weight` is in **hectograms** (1 kg = 10 hg).

In [ ]:
# I'm asking: can I pull the five fields for every Pokémon on the list, one request per Pokémon?
# The result is a Python list of rows — once this runs, I have the raw data for my table.
rows = []
for entry in pokemon_list:
    r = requests.get(entry["url"])
    r.raise_for_status()
    p = r.json()
    first_type = p["types"][0]["type"]["name"]
    rows.append(
        {
            "name": p["name"],
            "height": p["height"],
            "weight": p["weight"],
            "base_experience": p["base_experience"],
            "first_type": first_type,
        }
    )
len(rows)

## STEP 1e — Build the DataFrame `df`

In [ ]:
# I'm asking: how do I load my list of dicts into pandas so I can use column names like df["name"]?
# The result is df with 20 rows (Bulbasaur through Raticate in the default list) — not a single Pokémon.
df = pd.DataFrame(rows)
df

## STEP 2 — Inspect the dataset

Use two quick checks: preview rows, then column types and counts.

In [ ]:
# I'm asking: what do the first handful of rows look like?
# The result tells me names and numbers line up with what I pulled from the API.
df.head()

In [ ]:
# I'm asking: what dtypes did pandas pick, and does every column have 20 non-null values?
# The result tells me if anything failed to load or got the wrong type.
df.info()

## STEP 3 — Three analytical questions

Below: **type mix**, **strong vs weak by `base_experience`**, **average weight by type**, and a **missing-value check**.

In [ ]:
# Question 1: How many of these 20 Pokémon share each first type?
# The result tells me whether the sample is spread across types or skewed — useful before comparing groups.
df["first_type"].value_counts()

In [ ]:
# Question 2: Which Pokémon have high base_experience (say above the middle of this group)?
# The result highlights the “stronger” ones in this slice — good for comparing names and types.
median_xp = df["base_experience"].median()
tough = df[df["base_experience"] >= median_xp].sort_values("base_experience", ascending=False)
tough[["name", "first_type", "base_experience", "weight"]]

In [ ]:
# Question 3: For each first type, what is the average weight (hectograms) in this sample?
# The result compares types on size — only a few Pokémon per type here, so it’s an exploratory average.
df.groupby("first_type")["weight"].mean().round(1)

In [ ]:
# Extra check: Are any columns missing values after the API calls?
# The result should be 0 for every column if nothing broke — I trust group means more after this.
df.isnull().sum()

## STEP 4 — Simple visualization

A labeled bar chart makes the **type mix** from `value_counts()` easy to see at a glance.

In [ ]:
import matplotlib.pyplot as plt

type_counts = df["first_type"].value_counts().sort_index()
ax = type_counts.plot(kind="bar", color="steelblue", figsize=(7, 4))
ax.set_title("How many Pokémon in this sample (n=20) have each first type?")
ax.set_xlabel("First type")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()